In [ ]:
# 转换音频文件为 mp3 格式，并设置比特率为 128kbps
import subprocess
from pathlib import Path

# 配置
input_dir = Path(r"")  # 替换为你的路径
SUPPORTED_EXTS = [".wav", ".flac", ".m4a", ".aac", ".opus", ".mp3", ".wma"]
BITRATE = "128k"

def get_mp3_bitrate(file: Path) -> str:
    """返回 mp3 文件的比特率（如 '128k'）"""
    result = subprocess.run([
        "ffprobe",
        "-v", "error",
        "-select_streams", "a:0",
        "-show_entries", "stream=bit_rate",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(file)
    ], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    try:
        bitrate_bits = int(result.stdout.strip())
        kbps = int(round(bitrate_bits / 1000))
        return f"{kbps}k"
    except:
        return ""

def convert_to_mp3(file: Path):
    temp_output = file.with_suffix(".tmp.mp3")
    final_output = file.with_suffix(".mp3")

    print(f"🔄 转换中: {file.name}")
    subprocess.run([
        "ffmpeg",
        "-y",
        "-i", str(file),
        "-vn",
        "-ar", "44100",
        "-ac", "2",
        "-b:a", BITRATE,
        "-c:a", "libmp3lame",
        str(temp_output)
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    try:
        file.unlink()  # 删除原文件
        temp_output.rename(final_output)
        print(f"✅ 已转换: {final_output.name}")
    except Exception as e:
        print(f"❌ 错误: {file.name} → {e}")

def process_all():
    for file in input_dir.rglob("*"):
        if not file.is_file() or file.suffix.lower() not in SUPPORTED_EXTS:
            print(f"not file music: {file}")
            continue

        if file.suffix.lower() == ".mp3":
            bitrate = get_mp3_bitrate(file)
            if bitrate == BITRATE:
                print(f"✔️ 保留（已是128kbps）: {file.name}")
                continue
        convert_to_mp3(file)
    print("\n🎉 所有音频处理完成！")

if __name__ == "__main__":
    process_all()
